In [ ]:
# =====================================================================
# SECTION 1: Data Loading & Preprocessing
# =====================================================================

import numpy as np
import pandas as pd

# Load the dataset
df = pd.read_csv('Q1_Sales_Data.csv')

# Preview the first few rows
df.head()

# Check missing values
df['Revenue'].isna().sum()

# Handle missing values
df['Sales'] = df['Sales'].fillna(df['Sales'].mean())  # Fill missing values with the mean of the column
df['Revenue'] = df['Revenue'].fillna(df['Revenue'].mean())  # Fill missing values with the mean of the column

# Convert 'Date' column to datetime
df['Date'] = pd.to_datetime(df['Date'])

# Extract Month and Year
df['Month'] = df['Date'].dt.month
df['Year'] = df['Date'].dt.year

# Remove duplicates
df.drop_duplicates(inplace=True)

df
df.isna().sum()

In [ ]:
# =====================================================================
# SECTION 2: Exploratory Data Analysis (EDA) & Summary Stats
# =====================================================================

# Summary statistics for numerical columns
summary_stats = df[['Sales', 'Revenue', 'Quantity']].describe()
summary_stats

# Correlation matrix
correlation_matrix = df[['Sales', 'Revenue', 'Quantity']].corr()
correlation_matrix

import seaborn as sns
import matplotlib.pyplot as plt

# Heatmap of correlation matrix
plt.figure(figsize=(10, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
# =====================================================================
# SECTION 3: Sales Analysis & Product Performance
# =====================================================================

# Top 5 products by revenue
top_products = df.groupby('Product')['Revenue'].sum().nlargest(5)
top_products

# Plot the bar chart
top_products.plot(kind='bar', color='teal')
plt.title('Top 5 Products by Revenue')
plt.xlabel('Product')
plt.ylabel('Revenue')
plt.show()

# Monthly sales trend
monthly_sales = df.groupby('Month')['Sales'].sum()
monthly_sales

# Plot the line chart
plt.figure(figsize=(10, 6))
monthly_sales.plot(kind='line', marker='o')
plt.title('Monthly Sales Trend in Q1')
plt.xlabel('Month')
plt.ylabel('Sales')

# Annotate highest and lowest months
max_month = monthly_sales.idxmax()
min_month = monthly_sales.idxmin()

plt.annotate(
    f'Highest: {monthly_sales[max_month]}',
    xy=(max_month, monthly_sales[max_month]),
    xytext=(max_month, monthly_sales[max_month] + 1000),
    arrowprops=dict(arrowstyle='->', color='green'),
    color='green'
)

plt.annotate(
    f'Lowest: {monthly_sales[min_month]}',
    xy=(min_month, monthly_sales[min_month]),
    xytext=(min_month, monthly_sales[min_month] - 1000),
    arrowprops=dict(arrowstyle='->', color='red'),
    color='red'
)

plt.show()

In [ ]:

# =====================================================================
# SECTION 4: Regional Analysis
# =====================================================================

# Sales by region
region_sales = df.groupby('Region')['Revenue'].sum()
region_sales

# Plot the bar chart
region_sales.plot(kind='bar', color='purple')
plt.title('Total Revenue per Region')
plt.xlabel('Region')
plt.ylabel('Revenue')
plt.show()

# Identify region with highest sales growth
growth_region = region_sales.idxmax()
growth_value = region_sales.max()
print(f"The region with the highest sales is {growth_region} with a revenue of {growth_value}.")

In [ ]:
# =====================================================================
# SECTION 5: Customer Behavior & Segmentation
# =====================================================================

# Customer Behavior Classification
customer_data = df.groupby('CustomerID').agg({
    'Sales': 'count',
    'Revenue': 'mean'
})
customer_data = customer_data.rename(columns={'Sales': 'PurchaseFrequency', 'Revenue': 'AvgOrderValue'})
customer_data

# Manual classification
def classify_customer(row):
    if row['AvgOrderValue'] >= 500 and row['PurchaseFrequency'] >= 5:
        return 'High Value'
    elif row['AvgOrderValue'] >= 200:
        return 'Medium Value'
    else:
        return 'Low Value'

# Apply the function to classify customers
customer_data['Category'] = customer_data.apply(classify_customer, axis=1)

# Count the number of customers in each category
category_counts = customer_data['Category'].value_counts()
category_counts

customer_data

# Scatter plot
plt.figure(figsize=(10, 6))

for label, group in customer_data.groupby('Category'):
    print(f"{label} customers: {len(group)}")
    plt.scatter(group['PurchaseFrequency'], group['AvgOrderValue'], label=label)

plt.xlabel('Purchase Frequency')
plt.ylabel('Average Order Value')
plt.title('Customer Value Segmentation')
plt.legend()
plt.show()